# Task 2B - 04 An exact solver
The first solver enumerates every two-trip day plan per reefer and searches the combinations by branch-and-bound. It stops at a 3,000,000-node limit (about 5 minutes on this machine), so its answer is *best found*, **not proven optimal**. Here we replace the search by exact optimisation.

## Formulation
**Stage R - reefers and chilled orders (MILP, solved with HiGHS).**
A *column* is "reefer *r* runs trip *S*", where *S* is a set of chilled orders from one district that fits the reefer's volume and weight and the 270-minute budget. Binary `x[r,S]`; constraints: each order in at most one trip; per reefer at most 2 trips and trip minutes <= 270. Priority is a list of **lexicographic levels**: each level is maximised, then fixed, then the next is optimised.

**Stage A - all other orders on ambient vehicles (exact subset DP).**
Every non-chilled order that can be served is served; for each brand+district group we find the cheapest split into feasible trips by dynamic programming over order subsets (at most 14 orders), minimising (1) stops arriving after their window closes, (2) vehicles used, (3) minutes. Trucks are interchangeable except for capacity, so trips are planned on the largest truck and then matched to concrete trucks (verified).

The two stages share no vehicles or orders, so combining their lexicographic optima gives the lexicographic optimum of the whole problem.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.append('../../tools/task2b')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json, time
from task2b_common import *
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 120)
plt.rcParams.update({'figure.dpi': 90, 'axes.grid': True, 'grid.alpha': .25})
for d in (METRICS, FIGURES, INTERIM): d.mkdir(parents=True, exist_ok=True)
scn, fleet, veh, dt, al = load_tables()
sub_team = pd.read_csv(SUBMISSION)            # the teammate's submitted allocation (read-only)
import task2b_exact as X
S = X.S


## 1. Stage R: the reefer problem

In [2]:
vehicles = list(S.AVAILABLE['S1']); scope = list(S.ORDERS.values())
reefers = [v for v in vehicles if v.temp == 'reefer']; chilled = [o for o in scope if o.chilled]
plan_f, lv_f, sec_f, n_f = X.reefer_stage(reefers, chilled, 'weighted', scope, late_aware=False)
print('FAST model   : %5d columns (reefer x one feasible trip) | solve %.1fs' % (n_f, sec_f))
for lv in lv_f: print('     level:', lv[1], '=', round(lv[2], 3))
plan_r, lv_r, sec_r, n_cols = X.reefer_stage(reefers, chilled, 'weighted', scope)
print('\nCLOCK-AWARE  : %5d columns (a reefer day plan = ordered pair of trips) | solve %.1fs' % (n_cols, sec_r))
for lv in lv_r: print('     level:', lv[1], '=', round(lv[2], 3))

FAST model   :   288 columns (reefer x one feasible trip) | solve 2.5s
     level: priority value (weighted score) = 539.723
     level: reefer trips (fewer is better) = 8.0
     level: reefer minutes (fewer is better) = 821.0



CLOCK-AWARE  :  9740 columns (a reefer day plan = ordered pair of trips) | solve 51.9s
     level: priority value (weighted score) = 539.723
     level: reefer stops arriving after window close (fewer is better) = 3.0
     level: reefer trips (fewer is better) = 8.0
     level: reefer minutes (fewer is better) = 821.0


**Result.** The exact optimum of the first solver's own objective is **539.723** - exactly the value its branch-and-bound found, in both models. So the first solver's reefer plan *value* was already optimal; what was missing was the **proof** (HiGHS reports "optimal" only when the gap is closed) and a much faster way to get it (about 3 s for the fast model vs about 5 min).

The second, **clock-aware** model goes one step further: among all plans with that same optimal value it picks the one whose reefer stops arrive after their windows close least often, because every column is a whole day plan whose clock is exact. It costs about 50 s, so sensitivity sweeps (notebook 05) use the fast model, which gives the same served set.

In [3]:
rows = []
for vid, ts in sorted(plan_r.items()):
    for k, tr in enumerate(ts, 1):
        rows.append((vid, S.VEHICLES_ALL[vid].type, k, tr[0].district, len(tr), round(sum(o.m3 for o in tr), 1), S.trip_minutes(tr), ' '.join(o.ref for o in tr)))
print('clock-aware reefer plan (weighted policy), trips listed in the order the plan runs them')
pd.DataFrame(rows, columns=['reefer', 'type', 'trip', 'district', 'stops', 'm3', 'trip_min', 'orders'])

clock-aware reefer plan (weighted policy), trips listed in the order the plan runs them


,reefer,type,trip,district,stops,m3,trip_min,orders
0,VEH003,truck,1,Colombo,1,10.1,39.0,S1-009
1,VEH003,truck,2,Kurunegala,3,25.0,210.0,S1-071 S1-073 S1-075
2,VEH006,truck,1,Gampaha,3,28.7,101.0,S1-031 S1-035 S1-038
3,VEH006,truck,2,Colombo,4,31.5,110.0,S1-007 S1-012 S1-016 S1-021
4,VEH007,truck,1,Gampaha,3,19.3,100.0,S1-028 S1-033 S1-041
5,VEH007,truck,2,Kalutara,3,16.0,134.0,S1-046 S1-048 S1-051
6,VEH036,van,1,Colombo,2,4.3,64.0,S1-001 S1-003
7,VEH036,van,2,Colombo,2,5.7,63.0,S1-005 S1-014


## 2. Stage A: window-aware planning of everything else

In [4]:
ambient = [v for v in vehicles if v.temp == 'ambient']; other = [o for o in scope if not o.chilled and o.ref != 'S1-078']
plan_a, lv_a, sec_a, n_trips = X.ambient_stage(ambient, other, 'weighted', scope)
print('solve time %.2fs' % sec_a)
for lv in lv_a: print('  ', lv[1], '=', lv[2])
rows = []
for vid, ts in sorted(plan_a.items()):
    for k, tr in enumerate(ts, 1):
        rows.append((vid, S.VEHICLES_ALL[vid].type, k, tr[0].brand, tr[0].district, len(tr), round(sum(o.m3 for o in tr), 1), S.trip_minutes(tr)))
amb = pd.DataFrame(rows, columns=['vehicle', 'type', 'trip', 'brand', 'district', 'stops', 'm3', 'trip_min']); amb

solve time 0.61s
   ambient orders served (all serviceable ones) = 58.0
   stops arriving after window close = 0.0
   ambient trips planned = 17.0
   vehicles used = 11.0
   trip minutes = 3155.0


,vehicle,type,trip,brand,district,stops,m3,trip_min
0,VEH008,truck,1,Fresh,Kalutara,7,17.9,243.0
1,VEH008,truck,2,Style,Galle,2,19.7,217.0
2,VEH009,truck,1,Fresh,Colombo,8,29.9,202.0
3,VEH009,truck,2,Tech,Colombo,3,12.8,193.0
4,VEH013,truck,1,Fresh,Gampaha,8,28.8,222.0
5,VEH015,truck,1,Fresh,Galle,6,15.3,242.0
6,VEH015,truck,2,Style,Kurunegala,1,9.4,173.0
7,VEH017,truck,1,Fresh,Matara,4,11.3,227.0
8,VEH017,truck,2,Style,Matara,1,7.5,183.0
9,VEH018,truck,1,Fresh,Kurunegala,4,9.0,244.0


**Finding.** With 24 ambient vehicles available and only about 17 trips needed, nothing forces a long trip: the planner splits busy districts across several trucks so that **no stop arrives after its window closes**. (The first solver ran two 10-stop trips of 249-270 minutes while 15 ambient vehicles stayed in the depot, which put 4 stops past their windows.)

## 3. Full solve, both policies

In [5]:
res = {}
for pol in ('weighted', 'lexicographic'):
    t = time.time(); res[pol] = X.solve_exact(pol); print(pol, 'solved in %.1fs' % (time.time() - t))
pd.DataFrame({p: X.kpis(r) for p, r in res.items()})

weighted solved in 53.6s


lexicographic solved in 93.5s


,weighted,lexicographic
served,79.00,77.00
deferred,6.00,8.00
m3,328.30,313.83
chilled_served,21.00,19.00
chilled_m3,140.72,126.25
deferred_yday_served,9.00,10.00
deferred_yday_total,10.00,10.00
priority_value,1599.30,1590.80
vehicles,15.00,15.00
trips,25.00,25.00


In [6]:
# reference: the first solver's submitted allocation, same KPIs from the CSV
st, late_team = simulate_submission(sub_team, S)
srv = [S.ORDERS[r] for r in sub_team[sub_team.decision == 'served'].order_ref]
print('first solver (submitted file): served', len(srv), '| m3 %.1f' % sum(o.m3 for o in srv), '| chilled', sum(o.chilled for o in srv),
      '| vehicles', sub_team[sub_team.decision == 'served'].vehicle_id.nunique(), '| late stops (clock simulation):', late_team)

first solver (submitted file): served 79 | m3 328.3 | chilled 21 | vehicles 13 | late stops (clock simulation): 8


**Reading.** Solving the *same weighted score* exactly reproduces the first solver's service outcome (79 orders, 328.3 m3, 21 chilled) - confirming it was optimal for that objective - in about 4 seconds instead of 5 minutes, and with the ambient trips re-planned. The **lexicographic** policy is different: it serves all ten orders skipped yesterday, at the cost of two chilled orders. Notebook 05 explains that trade-off.

The few stops still marked late in the exact plans are *second trips of reefers*: with the booklet's 270-minute budget a second Fresh trip can start after 08:00 in a literal clock simulation (06 asserts that this is true of every remaining late stop). The booklet says its budgets already allow for the return journey and its feasibility rules do not include windows, so we report these stops rather than treat them as plan defects.